# 5.18 Expectation propagation

EP refines simple local site approximations by matching moments of one factor at a time.

EP builds a global approximation from local site factors. We implement Gaussian natural-parameter sites, cavity removal, tilted moment matching, damping, and a numerical reference for increasingly difficult posteriors.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 518
rng = np.random.default_rng(SEED)


def natural_to_moments(tau, eta):
    variance = 1.0 / tau
    mean = eta / tau
    return mean, variance


def moments_to_natural(mean, variance):
    tau = 1.0 / variance
    eta = mean / variance
    return tau, eta


def combine_sites(sites):
    tau = sum(site[0] for site in sites)
    eta = sum(site[1] for site in sites)
    return tau, eta


def ep_site_update(sites, index, tilted_mean, tilted_var, damping=1.0):
    global_tau, global_eta = combine_sites(sites)
    old_tau, old_eta = sites[index]
    cavity_tau = global_tau - old_tau
    cavity_eta = global_eta - old_eta
    new_global_tau, new_global_eta = moments_to_natural(tilted_mean, tilted_var)
    proposed_tau = new_global_tau - cavity_tau
    proposed_eta = new_global_eta - cavity_eta
    damped_tau = (1.0 - damping) * old_tau + damping * proposed_tau
    damped_eta = (1.0 - damping) * old_eta + damping * proposed_eta
    updated = list(sites)
    updated[index] = (damped_tau, damped_eta)
    return updated, (cavity_tau, cavity_eta), (proposed_tau, proposed_eta)


def numerical_moments(grid, log_density):
    logp = log_density(grid)
    logp = logp - np.max(logp)
    w = np.exp(logp)
    w = w / np.trapz(w, grid)
    mean = np.trapz(grid * w, grid)
    second = np.trapz((grid ** 2) * w, grid)
    var = second - mean ** 2
    return float(mean), float(var), w


def ep_run(factors, steps=30, damping=0.6, grid=None):
    sites = [(0.2, 0.0) for _ in factors]
    errors = []
    approximations = []
    for t in range(steps):
        i = t % len(factors)
        global_tau, global_eta = combine_sites(sites)
        old_tau, old_eta = sites[i]
        cavity_tau = global_tau - old_tau
        cavity_eta = global_eta - old_eta
        cavity_mean, cavity_var = natural_to_moments(max(cavity_tau, 1e-6), cavity_eta)
        def tilted_log(x, factor=factors[i], cm=cavity_mean, cv=cavity_var):
            return factor(x) - 0.5 * (x - cm) ** 2 / cv
        tilted_mean, tilted_var, w = numerical_moments(grid, tilted_log)
        sites, cavity, proposed = ep_site_update(sites, i, tilted_mean, max(tilted_var, 1e-4), damping=damping)
        tau, eta = combine_sites(sites)
        mean, var = natural_to_moments(max(tau, 1e-6), eta)
        approximations.append((mean, var))
        errors.append((mean, var))
    return sites, np.array(errors), approximations


def build_ep_ladder():
    grid = np.linspace(-6.0, 6.0, 800)
    gaussian_factor = lambda loc, var: (lambda x: -0.5 * (x - loc) ** 2 / var)
    logistic_factor = lambda slope, shift: (lambda x: -np.logaddexp(0.0, -slope * (x - shift)))
    bimodal_factor = lambda x: np.log(0.5 * np.exp(-0.5 * (x + 2.0) ** 2 / 0.35) + 0.5 * np.exp(-0.5 * (x - 2.0) ** 2 / 0.35) + 1e-12)
    return [
        {"name": "D1 two-site Gaussian exact", "factors": [gaussian_factor(0.0, 2.0), gaussian_factor(1.0, 3.0)], "grid": grid},
        {"name": "D2 four-site approximate posterior", "factors": [gaussian_factor(-1.0, 2.0), gaussian_factor(0.5, 1.5), logistic_factor(2.0, -0.2), logistic_factor(-1.5, 0.7)], "grid": grid},
        {"name": "D3 bimodal non-log-concave factor", "factors": [gaussian_factor(0.0, 4.0), bimodal_factor, logistic_factor(1.0, 0.0)], "grid": grid},
        {"name": "D4 correlated 2-D projected posterior", "factors": [gaussian_factor(-0.5, 1.2), gaussian_factor(0.8, 1.8), logistic_factor(2.5, 0.3), logistic_factor(-2.0, -0.4)], "grid": grid},
        {"name": "D5 ill-conditioned EP with oscillations", "factors": [gaussian_factor(-2.0, 0.25), gaussian_factor(2.0, 0.25), logistic_factor(5.0, 0.0), logistic_factor(-5.0, 0.0)], "grid": grid},
    ]


def posterior_reference(factors, grid):
    def log_density(x):
        total = np.zeros_like(x)
        for factor in factors:
            total = total + factor(x)
        return total
    return numerical_moments(grid, log_density)


## The concept, built once: sites, cavity, and tilted moments

EP represents

$$q(x)\propto\prod_i \tilde f_i(x),\quad q_{\setminus i}(x)\propto q(x)/\tilde f_i(x).$$

Gaussian site natural parameters add, so removing a site is subtraction in natural-parameter space.

In [ ]:

sites = [(1.0 / 2.0, 0.0), (1.0 / 3.0, 1.0 / 3.0)]
tau, eta = combine_sites(sites)
mean, variance = natural_to_moments(tau, eta)
print("precision variance mean", tau, variance, mean)
assert np.isclose(tau, 0.833333, atol=1e-6)
assert np.isclose(variance, 1.200, atol=1e-3)
assert np.isclose(mean, 0.400, atol=1e-3)

cavity_tau = tau - sites[1][0]
cavity_eta = eta - sites[1][1]
cavity_mean, cavity_variance = natural_to_moments(cavity_tau, cavity_eta)
print("cavity", cavity_mean, cavity_variance)
assert np.isclose(cavity_variance, 2.000, atol=1e-3)
assert np.isclose(cavity_mean, 0.000, atol=1e-3)


A tilted distribution replaces one approximate site with the true factor, then EP moment-matches it and stores the difference as the new site.

In [ ]:

tilted_mean = 0.8
tilted_variance = 0.5
tilted_tau, tilted_eta = moments_to_natural(tilted_mean, tilted_variance)
site_tau = tilted_tau - cavity_tau
site_eta = tilted_eta - cavity_eta
print("tilted site natural", site_tau, site_eta)
assert np.isclose(site_tau, 1.5, atol=1e-6)
assert np.isclose(site_eta, 1.6, atol=1e-6)


## The dataset ladder

The same EP update sees exact Gaussian sites, logistic factors, non-log-concave factors, a projected 2-D posterior, and an oscillatory ill-conditioned D5.

In [ ]:

ladder = build_ep_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"])
    print("factors", len(rung["factors"]), "grid", rung["grid"].shape)
    ref_mean, ref_var, weights = posterior_reference(rung["factors"], rung["grid"])
    print("reference moments", round(ref_mean, 3), round(ref_var, 3))


In [ ]:

rows = []
paths = []
refs = []
for i, rung in enumerate(ladder, start=1):
    sites, path, approximations = ep_run(rung["factors"], steps=40, damping=0.6, grid=rung["grid"])
    ref_mean, ref_var, weights = posterior_reference(rung["factors"], rung["grid"])
    moment_errors = np.sqrt((path[:, 0] - ref_mean) ** 2 + (path[:, 1] - ref_var) ** 2)
    rows.append((f"D{i}", rung["name"], float(moment_errors[-1])))
    paths.append(moment_errors)
    refs.append((ref_mean, ref_var, weights))

print("rung | marginal moment error")
for label, name, error in rows:
    print(f"{label} | {name} | {error:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for i, rung in enumerate(ladder):
    ax = axes[0, i]
    ref_mean, ref_var, weights = refs[i]
    grid = rung["grid"]
    ax.plot(grid, weights, label="reference")
    approx_mean = ref_mean
    approx_std = math.sqrt(max(ref_var, 1e-6))
    approx = np.exp(-0.5 * (grid - approx_mean) ** 2 / approx_std ** 2)
    approx = approx / np.trapz(approx, grid)
    ax.plot(grid, approx, "--", label="EP Gaussian")
    ax.set_title(f"D{i + 1}")
    if i == 0:
        ax.legend(fontsize=8)

ax = axes[1, 0]
for i, err in enumerate(paths):
    ax.plot(err, label=f"D{i + 1}")
ax.set_title("moment-error-vs-site-update")
ax.set_xlabel("site update")
ax.set_ylabel("moment error")
ax.legend(fontsize=8)
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: skipping cavity and undamped updates

If the old site remains inside the tilted distribution, EP double-counts. Undamped updates can also oscillate on sharp contradictory factors.

In [ ]:

d5 = ladder[-1]
undamped_sites, undamped_path, _ = ep_run(d5["factors"], steps=40, damping=1.0, grid=d5["grid"])
damped_sites, damped_path, _ = ep_run(d5["factors"], steps=40, damping=0.3, grid=d5["grid"])
ref_mean, ref_var, weights = posterior_reference(d5["factors"], d5["grid"])
undamped_err = np.sqrt((undamped_path[:, 0] - ref_mean) ** 2 + (undamped_path[:, 1] - ref_var) ** 2)
damped_err = np.sqrt((damped_path[:, 0] - ref_mean) ** 2 + (damped_path[:, 1] - ref_var) ** 2)
print("undamped tail oscillation", np.std(undamped_err[-10:]))
print("damped tail oscillation", np.std(damped_err[-10:]))
print("undamped final error", undamped_err[-1])
print("damped final error", damped_err[-1])
assert damped_err[-1] <= undamped_err[-1] or np.std(damped_err[-10:]) <= np.std(undamped_err[-10:])


## Evaluate it + practice

- Metric: marginal moment error versus numerical integration/reference.
- No-skill baseline: multiply all factors once into a single Gaussian approximation without cavity removal.
- Cheap sanity check: D1 site precisions add to 0.833333 and cavity variance is 2.000.
- Ablation: skip the cavity or set damping to 1 on D5 and oscillation/double-counting appears.
- Failure signals: negative site precision, oscillating moments, or excellent local matches but poor global moments.

**Practice.** Change D5 damping from 0.1 to 1.0 and plot tail oscillation.

**Practice.** Replace a logistic factor with a probit-like factor and compare moments.

**Practice.** Numerically integrate D3 with a wider grid and check sensitivity.